# Import and setup

In [ ]:
%load_ext autoreload
%matplotlib inline
%autoreload 2

import numpy as np
import matplotlib.pyplot as plt
from copy import copy
import qickdawg as qd
from scipy.optimize import curve_fit

# ===== EDIT THIS: Set your RFSoC IP address =====
RFSOC_IP = '192.168.0.103'

qd.start_client(RFSOC_IP)
print(f"Connected to RFSoC at {RFSOC_IP}")

In [ ]:
default_config = qd.NVConfiguration()

# ADC channel: 0 or 1 (run diagnostic above if unsure which has your photodiode)
# 0 for ADC_D and 1 for ADC_C
default_config.adc_channel = 0
default_config.mw_channel = 1
default_config.mw_nqz = 1
default_config.mw_gain = 5000
default_config.laser_gate_pmod = 0
default_config.relax_delay_tns = 500
default_config.readout_integration_tus = 213  # max ~106.7 µs; auto-sets _treg and _tns

print("Default configuration:")
print(f"  ADC channel:     {default_config.adc_channel}")
print(f"  MW channel:      {default_config.mw_channel}")
print(f"  MW gain:         {default_config.mw_gain}")
print(f"  Relax delay:     {default_config.relax_delay_tns:.0f} ns")
print(f"  Readout int.:    {default_config.readout_integration_tus:.2f} µs  ({default_config.readout_integration_treg} treg)")

# ODMR

In [ ]:
# =============================================================================
# ODMR sweep — edit ALL parameters below, then run this cell once.
# Builds config, runs LockinODMR, saves full sweep to CSV (MW on / MW off / contrast).
# =============================================================================
import pandas as pd
from pathlib import Path
from datetime import datetime
import importlib

from IPython.display import FileLink, display

# --- Microwave & timing ---
ODMR_MW_GAIN = 10500              # 0–32767; ↑ if contrast weak, ↓ if saturated
ODMR_PRE_INIT = True             # MW+laser prepulse before sweep
ODMR_REPS = 10                   # averages per frequency point
ODMR_RELAX_DELAY_TREG = 500     # delay register units between MW on/off segments 100000 

# --- ADC readout integration (photodiode PL averaging time) ---
ODMR_READOUT_INTEGRATION_TUS = 213   # or set e.g. 50.0 (microseconds)

# --- Frequency sweep (MHz) ---
MW_FREQ_START_MHZ = 2600
MW_FREQ_STOP_MHZ = 3100
MW_FREQ_STEP_MHZ = 1             # step size (MHz); narrow range once peaks are known
# Instead of delta, you can use a fixed point count, e.g.:
#   config_odmr.add_linear_sweep("mw", "fMHz", start=2700, stop=3000, nsweep_points=301)
# (then comment out the add_linear_sweep call in the build block below and paste there)

# --- CSV export ---
SAVE_ODMR_CSV = True             # set False to only acquire (still defines d, prog_odmr for plots)

# --- If reference should be flat but shows dips: try True (explicit low-gain pulse in ref. window) ---
ODMR_REFERENCE_ZERO_GAIN_PULSE = False
ODMR_REFERENCE_PULSE_GAIN = 1       # use 1 if gain 0 faults; increase only if needed
# --- Only if reference is flat and signal is not (mis-labeled shots), try True ---
ODMR_LOCKIN_SWAP_SIGNAL_REFERENCE = False

# --- Build configuration (inherits adc_channel, mw_channel, laser from default_config) ---
config_odmr = copy(default_config)
config_odmr.odmr_reference_zero_gain_pulse = ODMR_REFERENCE_ZERO_GAIN_PULSE
config_odmr.odmr_reference_pulse_gain = ODMR_REFERENCE_PULSE_GAIN
config_odmr.odmr_lockin_swap_signal_reference = ODMR_LOCKIN_SWAP_SIGNAL_REFERENCE
config_odmr.readout_integration_tus = ODMR_READOUT_INTEGRATION_TUS
config_odmr.mw_gain = ODMR_MW_GAIN
config_odmr.pre_init = ODMR_PRE_INIT
config_odmr.reps = ODMR_REPS
config_odmr.relax_delay_treg = ODMR_RELAX_DELAY_TREG
config_odmr.add_linear_sweep(
    "mw", "fMHz",
    start=MW_FREQ_START_MHZ,
    stop=MW_FREQ_STOP_MHZ,
    delta=MW_FREQ_STEP_MHZ,
)

print("ODMR configuration:")
print(f"  ADC channel: {config_odmr.adc_channel}  MW channel: {config_odmr.mw_channel}  gain: {config_odmr.mw_gain}")
print(f"  Sweep: {config_odmr.mw_start_fMHz:.3f} → {config_odmr.mw_end_fMHz:.3f} MHz  ({config_odmr.nsweep_points} pts)")
print(f"  reps: {config_odmr.reps}  est. time: {qd.LockinODMR(config_odmr).total_time():.1f} s")

prog_odmr = qd.LockinODMR(config_odmr)
d = prog_odmr.acquire(progress=True)
print("Acquisition done.")

if SAVE_ODMR_CSV:
    odmr_df = pd.DataFrame(
        {
            "frequency_MHz": np.asarray(d.frequencies, dtype=float),
            "photoluminescence_mw_on_ADC": np.asarray(d.signal, dtype=float),
            "photoluminescence_mw_off_ADC": np.asarray(d.reference, dtype=float),
            "contrast_ADC": np.asarray(d.contrast, dtype=float),
            "contrast_percent": np.asarray(d.contrast_percent, dtype=float),
        }
    )
    ts = datetime.now().strftime("%Y%m%d_%H%M%S")
    csv_path = Path.cwd() / f"odmr_sweep_{ts}.csv"
    odmr_df.to_csv(csv_path, index=False)
    LAST_ODMR_CSV = csv_path
    LAST_ODMR_DF = odmr_df.copy()
    print(f"CSV: {len(odmr_df)} rows → {csv_path.resolve()}")
    display(FileLink(csv_path.name))

In [ ]:
qd.LockinODMR.plot_sequence(config_odmr)

# Plot ODMR spectrum with subplots for MW On, MW Off, and Contrast
fig, axes = plt.subplots(3, 1, figsize=(12, 12), sharex=True)

# Plot MW On (signal)
axes[0].plot(d.frequencies, d.signal, label='MW On (signal)', color='steelblue')
axes[0].set_ylabel('PL Intensity (ADC units)')
axes[0].set_title('ODMR: MW On (signal)')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Plot MW Off (reference)
axes[1].plot(d.frequencies, d.reference, label='MW Off (reference)', color='orange')
axes[1].set_ylabel('PL Intensity (ADC units)')
axes[1].set_title('ODMR: MW Off (reference)')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# Plot Contrast
axes[2].plot(d.frequencies, d.contrast, label='Contrast (ADC)', color='darkred')
axes[2].set_xlabel('Frequency (MHz)')
axes[2].set_ylabel('Contrast (ADC units)')
axes[2].set_title('ODMR: Contrast')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Detect and mark ODMR dips from frequency (column 1) vs MW-off PL (column 3)
import pandas as pd
from pathlib import Path
from scipy.signal import find_peaks as scipy_find_peaks, savgol_filter

ODMR_CSV_PATH = None
# ODMR_CSV_PATH = Path("/Users/ckasemtantikul/Documents/PhD/NV Compact Magnetometer/Initial Test/odmr_sweep_20260320_132708.csv")
# Set ODMR_CSV_PATH to a saved CSV to analyze first-column frequency vs third-column PL.
# Leave it as None to analyze the current in-memory sweep (`d.reference`).


def load_odmr_trace(csv_path=None):
    if csv_path is None:
        return (
            np.asarray(d.frequencies, dtype=float),
            np.asarray(d.reference, dtype=float),
            'MW-off reference',
            'Current acquisition (`d.reference`)'
        )

    odmr_df = pd.read_csv(Path(csv_path))
    if {'frequency_MHz', 'photoluminescence_mw_off_ADC'}.issubset(odmr_df.columns):
        x = odmr_df['frequency_MHz'].to_numpy(dtype=float)
        y = odmr_df['photoluminescence_mw_off_ADC'].to_numpy(dtype=float)
        y_label = 'photoluminescence_mw_off_ADC'
    else:
        x = odmr_df.iloc[:, 0].to_numpy(dtype=float)
        y = odmr_df.iloc[:, 2].to_numpy(dtype=float)
        y_label = odmr_df.columns[2]

    return x, y, y_label, str(Path(csv_path).resolve())


def find_odmr_dips(
    frequencies,
    signal,
    n_dips=4,
    prominence_factor=0.4,
    min_prominence=6.0,
    smooth_window=7,
    polyorder=2,
    min_distance_pts=3,
    freq_min=None,
    freq_max=None,
):
    """Find ODMR transmission dips (local minima) in the selected PL trace within a frequency range."""
    x = np.asarray(frequencies, dtype=float)
    y = np.asarray(signal, dtype=float)

    valid = np.isfinite(x) & np.isfinite(y)
    x = x[valid]
    y = y[valid]

    # Limit analysis to within freq_min and freq_max if given
    if freq_min is not None or freq_max is not None:
        idx_range = np.ones_like(x, dtype=bool)
        if freq_min is not None:
            idx_range &= x >= freq_min
        if freq_max is not None:
            idx_range &= x <= freq_max
        x = x[idx_range]
        y = y[idx_range]

    if len(x) < 5:
        raise ValueError('Need at least 5 finite points to detect ODMR dips.')

    w = int(max(5, smooth_window))
    if w % 2 == 0:
        w += 1
    if w >= len(y):
        w = len(y) - 1 if len(y) % 2 == 0 else len(y)

    if w < polyorder + 2:
        y_smooth = y.copy()
    else:
        y_smooth = savgol_filter(y, window_length=w, polyorder=min(polyorder, w - 2), mode='interp')

    trend = np.polyval(np.polyfit(x, y_smooth, 1), x)
    y_detr = y_smooth - trend

    prominence = max(float(min_prominence), np.std(y_detr) * float(prominence_factor))
    dip_idx, props = scipy_find_peaks(
        -y_detr,
        prominence=prominence,
        distance=max(1, int(min_distance_pts)),
    )

    if len(dip_idx) == 0:
        return dip_idx, y_smooth, y_detr, props, prominence, x, y

    prom_arr = props.get('prominences', np.zeros(len(dip_idx), dtype=float))
    keep = np.argsort(prom_arr)[-int(n_dips):]
    dip_idx = dip_idx[keep]
    dip_idx = dip_idx[np.argsort(x[dip_idx])]
    return dip_idx, y_smooth, y_detr, props, prominence, x, y


# Restrict frequency range for dip detection: 2700 to 3000 MHz
FREQ_MIN = 2700
FREQ_MAX = 3000

frequencies, trace, trace_label, trace_source = load_odmr_trace(ODMR_CSV_PATH)
dip_idx, trace_smooth, trace_detr, dip_props, used_prominence, x_range, y_range = find_odmr_dips(
    frequencies,
    trace,
    n_dips=8,
    prominence_factor=0.4,
    min_prominence=4.0,
    smooth_window=7,
    polyorder=2,
    min_distance_pts=3,
    freq_min=FREQ_MIN,
    freq_max=FREQ_MAX,
)

plt.figure(figsize=(12, 4))

# Plot full raw and smoothed traces for visual context
plt.plot(frequencies, trace, color='green', linewidth=1.2, alpha=0.25, label='Raw trace (full)')
plt.plot(frequencies, savgol_filter(trace, window_length=7 if len(trace) > 7 else 5, polyorder=2 if len(trace) > 2 else 1, mode='interp'), color='seagreen', linewidth=1.0, alpha=0.15, label='Smoothed trace (full)')

# Plot zoomed-in range (2700-3000)
plt.plot(x_range, y_range, color='green', linewidth=1.2, alpha=0.75, label=f'Raw trace ({FREQ_MIN}-{FREQ_MAX} MHz)')
plt.plot(x_range, trace_smooth, color='seagreen', linewidth=2.0, label='Smoothed trace (range)')

if len(dip_idx) > 0:
    plt.plot(x_range[dip_idx], y_range[dip_idx], 'rv', markersize=10, label='Detected dips')
    for i, idx in enumerate(dip_idx):
        plt.annotate(
            f"{x_range[idx]:.1f} MHz",
            xy=(x_range[idx], y_range[idx]),
            xytext=(0, -14),
            textcoords='offset points',
            ha='center',
            va='top',
            fontsize=9,
            color='darkred',
        )

plt.ylabel('MW-off PL (ADC units)' if trace_label == 'MW-off reference' else trace_label)
plt.xlabel('Frequency (MHz)')
plt.title(f'ODMR Dip Detection in {FREQ_MIN}-{FREQ_MAX} MHz ({len(dip_idx)} found)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f'Trace source: {trace_source}')
print(f'Analyzed range: {FREQ_MIN} to {FREQ_MAX} MHz')
print('Using column 1 as x and column 3 as y when a CSV path is provided.')
print(f'Prominence threshold used: {used_prominence:.2f} ADC')

if len(dip_idx) == 0:
    print('No ODMR dips found in the specified range. Lower `min_prominence` or reduce `smooth_window`.')
else:
    print('Detected ODMR dips in selected range:')
    for i, idx in enumerate(dip_idx):
        print(f"  Dip {i+1}: {x_range[idx]:.2f} MHz  ({y_range[idx]:.4f} ADC)")

    deepest_idx = dip_idx[np.argmin(y_range[dip_idx])]
    print(f'Suggested resonance frequency (deepest dip in range): {x_range[deepest_idx]:.2f} MHz')


# Lockin

In [ ]:
# Step 1 — Generate the 16 parked frequencies from the most recent ODMR sweep.
# Uses nv_toolkit's _suggest_parked_frequencies (same algorithm as `mag-operator plan`).
import sys
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import FileLink, display

# Locate NV_Magnetometry_Software-main (sibling to NV Compact Magnetometer/Initial Test).
_search_roots = [Path.cwd(), *Path.cwd().parents]
NV_REPO_DIR = None
for _root in _search_roots:
    candidate = _root / "NV_Magnetometry_Software-main"
    if (candidate / "nv_toolkit" / "operator_cli.py").exists():
        NV_REPO_DIR = candidate
        break
if NV_REPO_DIR is None:
    raise FileNotFoundError("NV_Magnetometry_Software-main not found relative to notebook directory")
if str(NV_REPO_DIR) not in sys.path:
    sys.path.insert(0, str(NV_REPO_DIR))

from nv_toolkit.tui import (
    _load_operator_full_scan,
    _suggest_parked_frequencies,
    _write_parked_plan_csv,
    _estimate_bias_field_mT,
)

# Pick the ODMR CSV. Default: the one saved by the ODMR cell above (LAST_ODMR_CSV).
if "LAST_ODMR_CSV" not in globals():
    raise RuntimeError("Run the ODMR sweep cell first to set LAST_ODMR_CSV")
ODMR_CSV_FOR_PLAN = Path(LAST_ODMR_CSV)

freqs_mhz, measured, transition_centers = _load_operator_full_scan(ODMR_CSV_FOR_PLAN, "auto")
parked_plan = _suggest_parked_frequencies(freqs_mhz, measured, transition_centers)

# Persist the plan as the canonical handoff CSV.
PARKED_PLAN_CSV = ODMR_CSV_FOR_PLAN.with_name(f"parked_plan_{ODMR_CSV_FOR_PLAN.stem}.csv")
_write_parked_plan_csv(PARKED_PLAN_CSV, parked_plan)

# Try to estimate the bias field; fall back to None if fitting fails.
try:
    BIAS_MT = np.asarray(_estimate_bias_field_mT(freqs_mhz, measured, transition_centers), dtype=float)
except Exception as exc:
    BIAS_MT = None
    print(f"Bias estimation skipped: {exc}")

# Build a flat ordered list of 16 frequencies (the order the FPGA must measure).
MULTIPOINT_FREQS_MHZ = []
for entry in parked_plan:
    MULTIPOINT_FREQS_MHZ.append(float(entry.frequency_minus_mhz))
    MULTIPOINT_FREQS_MHZ.append(float(entry.frequency_plus_mhz))
MULTIPOINT_FREQS_MHZ = np.asarray(MULTIPOINT_FREQS_MHZ, dtype=float)

plan_df = pd.DataFrame(
    [
        {
            "block_index": i + 1,
            "transition_index": entry.transition_index,
            "center_mhz": entry.center_mhz,
            "linewidth_mhz": entry.linewidth_mhz,
            "f_minus_mhz": entry.frequency_minus_mhz,
            "f_plus_mhz": entry.frequency_plus_mhz,
            "slope_minus_per_mhz": entry.slope_minus_per_mhz,
            "slope_plus_per_mhz": entry.slope_plus_per_mhz,
        }
        for i, entry in enumerate(parked_plan)
    ]
)
print(f"ODMR source: {ODMR_CSV_FOR_PLAN}")
print(f"Parked plan written to: {PARKED_PLAN_CSV}")
print(f"Number of parked frequencies: {len(MULTIPOINT_FREQS_MHZ)}")
if BIAS_MT is not None:
    print(f"Estimated bias field (mT): {BIAS_MT[0]:.4f}, {BIAS_MT[1]:.4f}, {BIAS_MT[2]:.4f}")
display(plan_df)
display(FileLink(str(PARKED_PLAN_CSV)))


In [ ]:
# Step 2 — Acquire all 16 parked frequencies in ONE FPGA program upload.
# Avoids the ~5 s overhead of building 16 separate LockinODMR programs.
import sys
from copy import copy
from pathlib import Path
from time import perf_counter, time
import numpy as np
import pandas as pd
from IPython.display import FileLink, display

# Make the local Modules folder importable for multipoint_lockin_program.
_modules_dir = Path.cwd() / "Modules" if (Path.cwd() / "Modules").exists() else Path.cwd()
if str(_modules_dir) not in sys.path:
    sys.path.insert(0, str(_modules_dir))

import importlib
import multipoint_lockin_program
multipoint_lockin_program = importlib.reload(multipoint_lockin_program)
from multipoint_lockin_program import MultipointLockinODMR

# --- Acquisition parameters ---
MULTIPOINT_REPS_PER_BATCH = 1000
MULTIPOINT_N_BATCHES = 1
MULTIPOINT_OFF_RESONANCE_MHZ = 2650.0     # off-resonance reference frequency
MULTIPOINT_DATA_CSV = Path.cwd() / "multipoint_lockin_collected.csv"

# --- Build the multipoint config ---
cfg = copy(default_config)
cfg.multipoint_freqs_mhz = list(MULTIPOINT_FREQS_MHZ)
cfg.odmr_reference_offres_mhz = MULTIPOINT_OFF_RESONANCE_MHZ
# QickSweep needs >= 2 points; we add a trivial 2-point zero-span outer sweep.
cfg.mw_start_fMHz = float(MULTIPOINT_FREQS_MHZ[0])
cfg.mw_end_fMHz = float(MULTIPOINT_FREQS_MHZ[0])
cfg.nsweep_points = 1
cfg.reps = int(MULTIPOINT_REPS_PER_BATCH)
cfg.pre_init = getattr(default_config, "pre_init", True)

# --- Build ONE program (handles all 16 frequencies internally) ---
t_build_start = perf_counter()
prog = MultipointLockinODMR(cfg)
print(f"Built single MultipointLockinODMR program for {len(MULTIPOINT_FREQS_MHZ)} frequencies in {perf_counter() - t_build_start:.2f} s")
print(f"Estimated time per batch: {prog.total_time():.3f} s ({prog.time_per_rep()*1e3:.1f} ms/rep × {cfg.reps} reps)")

# --- Collect N batches, each batch = one acquire = one full sweep over all 16 freqs ---
rows = []
t0 = perf_counter()
for batch in range(int(MULTIPOINT_N_BATCHES)):
    t_acq_start = perf_counter()
    d_batch = prog.acquire(progress=False)
    acq_dt = perf_counter() - t_acq_start

    row = {
        "batch": batch,
        "time_s": perf_counter() - t0,
        "timestamp_epoch_s": time(),
        "acq_seconds": acq_dt,
    }
    # Record one column per parked frequency, in point_index order.
    for k, (freq_mhz, sig, ref) in enumerate(zip(d_batch.frequencies_mhz, d_batch.signal, d_batch.reference)):
        row[f"peak_{k+1:02d}"] = float(sig)
        row[f"peak_{k+1:02d}_ref"] = float(ref)
        row[f"peak_{k+1:02d}_freq_mhz"] = float(freq_mhz)
    rows.append(row)
    print(f"batch {batch+1}/{int(MULTIPOINT_N_BATCHES)}: acquired in {acq_dt:.3f} s")

df_multipoint = pd.DataFrame(rows)
df_multipoint.to_csv(MULTIPOINT_DATA_CSV, index=False)
print(f"\nSaved {len(df_multipoint)} batches to {MULTIPOINT_DATA_CSV}")
display(df_multipoint)
display(FileLink(str(MULTIPOINT_DATA_CSV)))


In [ ]:
# Step 3 — Reconstruct B-field vectors from collected parked-frequency data.
# Uses nv_toolkit's _compute_live_snapshot (same algorithm as `mag-operator reconstruct`).
import csv
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import FileLink, display

from nv_toolkit.tui import OperatorConfig, _compute_live_snapshot

# Inputs (set by the cells above)
_full_scan_path = ODMR_CSV_FOR_PLAN
_plan_csv_path = PARKED_PLAN_CSV
_collected_csv_path = MULTIPOINT_DATA_CSV

# The wide format expected by _compute_live_snapshot needs:
#   time_s + 16 intensity columns in point_index order (no _ref/_freq_mhz extras).
# We rewrite our richer collected CSV into a clean wide format here.
_wide_for_recon = _collected_csv_path.with_name(_collected_csv_path.stem + "_wide.csv")
_clean_df = pd.DataFrame({"time_s": df_multipoint["time_s"]})
for k in range(len(MULTIPOINT_FREQS_MHZ)):
    col = f"peak_{k+1:02d}"
    _clean_df[col] = df_multipoint[col]
_clean_df.to_csv(_wide_for_recon, index=False)

# Reuse the bias estimate from the plan cell (or recompute here).
_bias_mT = BIAS_MT if BIAS_MT is not None else np.zeros(3, dtype=float)

config = OperatorConfig(
    full_scan_path=_full_scan_path,
    parked_data_path=_wide_for_recon,
    parked_format="wide",
    bias_field_mT=np.asarray(_bias_mT, dtype=float),
    reference_freqs_mhz=np.asarray(freqs_mhz, dtype=float),
    reference_spectrum=np.asarray(measured, dtype=float),
    transition_centers_mhz=np.asarray(transition_centers, dtype=float),
    parked_plan=tuple(parked_plan),
    plan_csv_path=_plan_csv_path,
    wide_template_path=_plan_csv_path.with_name("parked_template_wide.csv"),
    long_template_path=_plan_csv_path.with_name("parked_template_long.csv"),
    poll_interval_s=0.0,
)
snapshot = _compute_live_snapshot(config)
if snapshot.status == "error":
    raise ValueError(snapshot.message)

# Output CSVs
VECTOR_CSV = _collected_csv_path.with_name("reconstructed_vector_rows.csv")
PROJECTION_CSV = _collected_csv_path.with_name("reconstructed_projection_rows.csv")

def _write_rows(path, rows):
    if not rows:
        path.write_text("")
        return
    fieldnames = []
    for r in rows:
        for key in r:
            if key not in fieldnames:
                fieldnames.append(str(key))
    with path.open("w", newline="") as h:
        writer = csv.DictWriter(h, fieldnames=fieldnames)
        writer.writeheader()
        for r in rows:
            writer.writerow({k: r.get(k, "") for k in fieldnames})

_write_rows(VECTOR_CSV, snapshot.vector_rows)
_write_rows(PROJECTION_CSV, snapshot.projection_rows)

print(f"status: {snapshot.status}")
print(f"samples: {snapshot.sample_count}  rank-ok: {snapshot.ok_count}")
print(f"vector CSV:     {VECTOR_CSV}")
print(f"projection CSV: {PROJECTION_CSV}")

if snapshot.vector_rows:
    display(pd.DataFrame(snapshot.vector_rows))
if snapshot.projection_rows:
    print("Projection rows (one per block per timestamp):")
    display(pd.DataFrame(snapshot.projection_rows))

display(FileLink(str(VECTOR_CSV)))
display(FileLink(str(PROJECTION_CSV)))
